# Wan 2.2 fence video (Kaggle free GPU)

**Before running:** In the right sidebar, set `Accelerator` to `GPU T4 x2` (or `GPU P100`) and turn `Internet` **On**. Then `Run All`.

This does NOT need the browser tab to stay open once you click **Save Version > Save & Run All (Commit)** -- it runs on Kaggle's servers in the background. The finished video appears in the notebook's **Output** tab when the commit finishes.

No image upload needed -- the 3 source images are fetched automatically from GitHub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
major, minor = (int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if (major, minor) < (2, 7):
    print('Upgrading torch...')
    !pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('If this ran, restart the kernel and re-run from the top.')

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt

In [ ]:
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git
!git clone --depth 1 https://github.com/stduhpf/ComfyUI--Wan22FirstLastFrameToVideoLatent.git
!pip install -q -r ComfyUI-GGUF/requirements.txt
%cd /kaggle/working/ComfyUI

In [ ]:
import os
import urllib.request

os.makedirs('models/unet', exist_ok=True)
os.makedirs('models/text_encoders', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)
os.makedirs('input', exist_ok=True)

def expected_size(url):
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req) as r:
        return int(r.headers.get('Content-Length', -1))

files_to_check = [
    ('models/unet/Wan2.2-TI2V-5B-Q8_0.gguf', 'https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q8_0.gguf'),
    ('models/text_encoders/umt5-xxl-encoder-Q8_0.gguf', 'https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q8_0.gguf'),
    ('models/vae/Wan2.2_VAE.safetensors', 'https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/VAE/Wan2.2_VAE.safetensors'),
]

for path, url in files_to_check:
    exp = expected_size(url)
    if os.path.exists(path) and os.path.getsize(path) == exp:
        print(f'{path}: already present ({exp} bytes) -- skipping download')
        continue
    print(f'Downloading {path} ...')
    os.system(f'wget -q --show-progress -O "{path}" "{url}"')
    actual = os.path.getsize(path)
    status = 'OK' if actual == exp else 'MISMATCH -- re-download this file!'
    print(f'{path}: {actual} / {exp} expected -- {status}')

In [ ]:
# Fetch the 3 source images from GitHub -- no manual upload needed.
!wget -q -O input/start_no_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/start%20frame.png"
!wget -q -O input/mid_pillars.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/start%20frame%20with%20fence.png"
!wget -q -O input/end_full_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/end%20frame%20with%20fence.png"
!ls -la input/
print('Images ready in input/')

In [ ]:
import subprocess, time, urllib.request

try:
    urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
    print('ComfyUI already running')
except Exception:
    proc = subprocess.Popen(
        ['python', 'main.py'],
        stdout=open('/kaggle/working/comfyui.log', 'w'), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
            print('ComfyUI is up')
            break
        except Exception:
            time.sleep(2)
    else:
        print('ComfyUI did not start -- check /kaggle/working/comfyui.log')
        !tail -n 60 /kaggle/working/comfyui.log

In [ ]:
import json, time, urllib.request

SERVER = "http://127.0.0.1:8188"
UNET = "Wan2.2-TI2V-5B-Q8_0.gguf"
CLIP = "umt5-xxl-encoder-Q8_0.gguf"
VAE = "Wan2.2_VAE.safetensors"

WIDTH = 1280
HEIGHT = 704
LENGTH = 61  # ~2.5s at 24fps per segment; two segments = ~5s total
STEPS = 30
CFG = 6.0
SHIFT = 5.0
SEED = 42

SEGMENT_A_PROMPT = (
    "aerial drone shot flying forward over farmland village at sunset, camera "
    "moving steadily forward, tall wooden fence pillars slowly descending "
    "straight down from the sky in perfect symmetrical rows, lowering gently "
    "and gradually like they are being placed by an unseen hand, touching "
    "down softly to plant themselves into the ground one after another "
    "forming a straight line of fence posts stretching into the distance, "
    "slow controlled descent, no falling or impact, magical stop-motion "
    "construction, cinematic lighting, photorealistic"
)
SEGMENT_B_PROMPT = (
    "aerial drone shot flying forward over farmland village at sunset, camera "
    "moving steadily forward, barbed wire unspooling and stretching itself "
    "between the wooden fence pillars, hooking onto each post one by one in "
    "sequence from near to far, wire strands pulling taut across the line of "
    "posts, magical stop-motion construction, cinematic lighting, photorealistic"
)
NEGATIVE_PROMPT = "blurry, low quality, distorted, flickering, artifacts, watermark, text, static camera, jerky motion, falling, dropping, crashing, impact, bouncing"

def build_graph(start_image, end_image, filename_prefix, positive_prompt):
    return {
        "unet_loader": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET}},
        "clip_loader": {"class_type": "CLIPLoaderGGUF", "inputs": {"clip_name": CLIP, "type": "wan"}},
        "vae_loader": {"class_type": "VAELoader", "inputs": {"vae_name": VAE}},
        "model_sampling": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["unet_loader", 0], "shift": SHIFT}},
        "positive": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["clip_loader", 0], "text": positive_prompt}},
        "negative": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["clip_loader", 0], "text": NEGATIVE_PROMPT}},
        "load_start": {"class_type": "LoadImage", "inputs": {"image": start_image}},
        "load_end": {"class_type": "LoadImage", "inputs": {"image": end_image}},
        "flf_latent": {
            "class_type": "Wan22FirstLastFrameToVideoLatent",
            "inputs": {
                "vae": ["vae_loader", 0], "width": WIDTH, "height": HEIGHT, "length": LENGTH,
                "batch_size": 1, "start_image": ["load_start", 0], "end_image": ["load_end", 0],
            },
        },
        "ksampler": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["model_sampling", 0], "positive": ["positive", 0], "negative": ["negative", 0],
                "latent_image": ["flf_latent", 0], "seed": SEED, "steps": STEPS, "cfg": CFG,
                "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1.0,
            },
        },
        # Tiled decode -- avoids the VRAM spike that killed the process on
        # Colab's T4 right after sampling finished.
        "vae_decode": {
            "class_type": "VAEDecodeTiled",
            "inputs": {
                "samples": ["ksampler", 0], "vae": ["vae_loader", 0],
                "tile_size": 256, "overlap": 64, "temporal_size": 32, "temporal_overlap": 8,
            },
        },
        "save_video": {
            "class_type": "SaveWEBM",
            "inputs": {"images": ["vae_decode", 0], "filename_prefix": filename_prefix, "codec": "vp9", "fps": 24.0, "crf": 20.0},
        },
    }

def queue_prompt(graph):
    data = json.dumps({"prompt": graph}).encode("utf-8")
    req = urllib.request.Request(f"{SERVER}/prompt", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

def wait_for_completion(prompt_id, poll_interval=10, timeout=7200):
    start = time.time()
    while time.time() - start < timeout:
        with urllib.request.urlopen(f"{SERVER}/history/{prompt_id}") as resp:
            hist = json.loads(resp.read())
        if prompt_id in hist:
            entry = hist[prompt_id]
            status = entry.get("status", {})
            if status.get("completed"):
                return entry
            if status.get("status_str") == "error":
                raise RuntimeError(f"Prompt {prompt_id} failed: {json.dumps(status, indent=2)}")
        elapsed = int(time.time() - start)
        if elapsed % 60 < poll_interval:
            print(f"  ...still running ({elapsed}s elapsed)")
        time.sleep(poll_interval)
    raise TimeoutError(f"Prompt {prompt_id} did not complete within {timeout}s")

def run_segment(name, start_image, end_image, filename_prefix, positive_prompt):
    print(f"=== Queuing segment: {name} ===")
    graph = build_graph(start_image, end_image, filename_prefix, positive_prompt)
    result = queue_prompt(graph)
    prompt_id = result["prompt_id"]
    print(f"  prompt_id={prompt_id}")
    entry = wait_for_completion(prompt_id)
    outputs = entry.get("outputs", {})
    video_info = outputs.get("save_video", {})
    print(f"  DONE: {json.dumps(video_info)}")
    return video_info

print('Ready. Run the next cell to generate.')

In [ ]:
seg_a = run_segment("A: no-fence -> pillars descending from sky", "start_no_fence.png", "mid_pillars.png", "fence_seg_a", SEGMENT_A_PROMPT)
seg_b = run_segment("B: pillars -> wire hooking on", "mid_pillars.png", "end_full_fence.png", "fence_seg_b", SEGMENT_B_PROMPT)
print("ALL SEGMENTS DONE")

In [ ]:
# Concatenate the two segments into one 5s video. Leaving the final file in
# /kaggle/working/ makes it show up in this notebook's Output tab once the
# commit finishes -- no explicit download call needed on Kaggle.
%cd /kaggle/working/ComfyUI/output
!ls -la *.webm

seg_a_file = [f for f in __import__('os').listdir('.') if f.startswith('fence_seg_a')][0]
seg_b_file = [f for f in __import__('os').listdir('.') if f.startswith('fence_seg_b')][0]

with open('concat_list.txt', 'w') as f:
    f.write(f"file '{seg_a_file}'\nfile '{seg_b_file}'\n")

!ffmpeg -y -f concat -safe 0 -i concat_list.txt -c:v libvpx-vp9 -crf 20 -b:v 0 fence_full_5s.webm
!ffmpeg -y -i fence_full_5s.webm -c:v libx264 -pix_fmt yuv420p -crf 18 /kaggle/working/fence_full_5s.mp4
print('Final video at /kaggle/working/fence_full_5s.mp4 -- check the Output tab after commit.')